<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/3_dise%C3%B1o_entrenamiento_evaluacion/3_1_modelo_x.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.1. Primer modelo X

## 0. Clonado de repositorio, importación de librerías y carga del dataset

### Clonado de repositorio e importación de librerías

In [4]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit

!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


In [5]:
import sys

#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")

!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
import seaborn as sns

Librería instalada: technical-analysis


### Carga del dataset mnq_intraday_data

In [6]:
def load_df():
    """
    Función para cargar un archivo Parquet desde el repositorio clonado
    """
    # Definir la URL del archivo Parquet en GitHub
    df_path = '/content/neural_profit/2_obtencion_preparacion_exploracion_datos/mnq_intraday_data.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(df_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [7]:
mnq_intraday = load_df()

## 1. Preparación dataset

### 1.1. Añadir target

In [8]:
def add_targets (df):
  df['target_return_30'] = df.groupby('date')['close'].transform(lambda x: np.log(x.shift(-30)) - np.log(x))
  return df

In [9]:
mnq_intraday = add_targets(mnq_intraday)

### 1.2. Añadir indicadores técnicos

In [10]:
def add_indicators(df=mnq_intraday, target='close' ):

  indicators_list= [
    'momentum_3',
    'momentum_10',
    'roc_5',
    'roc_20',
    'rsi_3',
    'rsi_7',
    'rsi_14',
    'stoch_k_20',
    'bb_percent_20_15',
    'bb_percent_30_20',
    'price_ema30',
    'atr_norm']


  def aplicar_por_dia (grupo):
        grupo = grupo.copy()

        # MOMENTUM y VELOCIDAD
        grupo['momentum_3'] = grupo[target].pct_change(3)
        grupo['momentum_10'] = grupo[target].pct_change(10)
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()

        # SOBRECOMPRA / SOBREVENTA (RSI)
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()

        stoch_20 = StochasticOscillator(high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3)
        grupo['stoch_k_20'] = stoch_20.stoch()

        # BOLLINGER BANDS
        grupo['bb_percent_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_percent_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()

        # TENDENCIA RELATIVA
        grupo['price_ema30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1

        # VOLATILIDAD
        atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=14)
        grupo['atr'] = atr.average_true_range()
        grupo['atr_norm'] = grupo['atr'] / grupo[target]
        grupo.drop(columns=['atr'], inplace=True)  #Elimino columna 'atr'
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, indicators_list

In [11]:
mnq_intraday, factores = add_indicators()

### 1.3. Añadir alpha factors de hipotesis

#### 1.3.1 Hipotesis 1

In [12]:
def add_reversion_momentum_factor(df: pd.DataFrame,
                          price_col: str = 'close',
                          date_col: str = 'date',
                          window: int = 30) -> pd.DataFrame:
    """
    Agrega una columna 'reversion_media_factor' que es el 'momentum_rev_z_30' al DataFrame, representando el alpha factor
    de reversión a la media basado en el z-score invertido de log-retornos a N minutos.

    Parámetros:
    - df: DataFrame de entrada con columnas de precios y fechas
    - price_col: nombre de la columna de precios (default='close')
    - date_col: nombre de la columna de fecha o jornada (default='date')
    - window: cantidad de minutos para calcular el momentum (default=30)

    Retorna:
    - DataFrame con la columna 'reversion_media_factor' añadida
    """
    df = df.copy()

    def calc_reversal(x):
        m = np.log(x) - np.log(x.shift(window))
        return -(m - m.mean()) / m.std()

    af_1 = 'reversal_momentum_factor'
    df[af_1] = df.groupby(date_col)[price_col].transform(calc_reversal)
    #df = df.dropna(subset=['momentum_rev_z_30'])

    return df, af_1

In [13]:
mnq_intraday, alpha_factor_1 = add_reversion_momentum_factor(mnq_intraday)

In [14]:
factores.append(alpha_factor_1)

#### 1.3.2. Hipotesis 2

In [15]:
def add_reversal_media_factor(df: pd.DataFrame, window: int = 30, price_col: str = 'close') -> pd.DataFrame:
    """
    Agrega al DataFrame un alpha factor basado en la hipótesis 2 (reversión a la media),
    llamado 'reversal_score_{window}', calculado como z-score normalizado por jornada.

    No guarda columnas intermedias.

    Parámetros:
    - df: DataFrame con columnas 'date' y una columna de precios (por defecto 'close')
    - window: Ventana para el cálculo del z-score (en minutos)
    - price_col: Nombre de la columna de precios

    Retorna:
    - El DataFrame original con una nueva columna 'reversal_score_{window}'
    """
    z_col = f'_tmp_z_{window}'
    reversal_col = f'reversal_media_factor'

    # Calcular z-score por jornada
    df[z_col] = df.groupby('date')[price_col].transform(
        lambda x: (x - x.rolling(window=window, min_periods=1).mean()) /
                  x.rolling(window=window, min_periods=1).std()
    )

    # Normalizar el z-score por jornada
    df[reversal_col] = df.groupby('date')[z_col].transform(
        lambda x: (x - x.mean()) / x.std()
    )

    # Eliminar columna temporal
    df.drop(columns=[z_col], inplace=True)

    return df, reversal_col

In [16]:
mnq_intraday, alpha_factor_2 = add_reversal_media_factor(mnq_intraday)

In [17]:
factores.append(alpha_factor_2)

#### 1.3.3. Hipotesis 3

In [18]:
def add_reversion_vol_momentum_factor(df: pd.DataFrame,
                                      price_col: str = 'close',
                                      volume_col: str = 'volume',
                                      date_col: str = 'date',
                                      window: int = 30) -> pd.DataFrame:
    """
    Agrega una columna 'reversion_vol_momentum_factor' al DataFrame,
    que representa un alpha factor de reversión basado en:
    - Momentum a N minutos
    - Relación volumen actual / volumen promedio N
    - Z-score del factor por jornada
    - Inversión de signo para capturar reversión

    Parámetros:
    - df: DataFrame de entrada con columnas de precios, volumen y fechas
    - price_col: nombre de la columna de precios (default='close')
    - volume_col: nombre de la columna de volumen (default='volume')
    - date_col: nombre de la columna de fecha o jornada (default='date')
    - window: ventana en minutos para calcular momentum y volumen promedio (default=30)

    Retorna:
    - DataFrame con la columna 'reversion_vol_momentum_factor' añadida
    """
    df = df.copy()

    # 1. Calcular el momentum
    df['momentum'] = df.groupby(date_col)[price_col].transform(
        lambda x: np.log(x) - np.log(x.shift(window))
    )

    # 2. Calcular volumen promedio intradía (rolling por jornada)
    df['vol_avg'] = df.groupby(date_col)[volume_col].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

    # 3. Relación volumen actual / volumen promedio
    df['vol_ratio'] = df[volume_col] / df['vol_avg']

    # 4. Alpha factor crudo: momentum × vol_ratio
    df['mom_vol'] = df['momentum'] * df['vol_ratio']

    # 5. Normalización diaria y reversión
    af_name = 'reversion_vol_momentum_factor'
    df[af_name] = df.groupby(date_col)['mom_vol'].transform(
        lambda x: -(x - x.mean()) / x.std()
    )

    # 6. Limpieza opcional de columnas intermedias
    df.drop(columns=['momentum', 'vol_avg', 'vol_ratio', 'mom_vol'], inplace=True)

    return df, af_name

In [19]:
mnq_intraday, alpha_factor_3 = add_reversion_vol_momentum_factor(mnq_intraday)

In [20]:
factores.append(alpha_factor_3)

#### 1.3.4. Hipotesis 4

In [21]:
def add_opening_exhaustion_factor30(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Calcular métricas de apertura por jornada
    opening_start = "08:00:00"
    opening_end = "09:30:00"

    def get_opening_metrics(x):
        high = x.between_time(opening_start, opening_end)['high'].max()
        low = x.between_time(opening_start, opening_end)['low'].min()
        opening_range = high - low
        midpoint = (high + low) / 2
        return pd.Series({'range': opening_range, 'midpoint': midpoint})

    opening_df = df.groupby('date').apply(get_opening_metrics)

    # Calcular factor raw (solo después de 09:30)
    df = df.join(opening_df.rename(columns={'range': 'opening_range', 'midpoint': 'opening_midpoint'}), on='date')
    df['after_opening'] = df.index >= df.index.normalize() + pd.Timedelta("09:30:00")

    df['factor_raw'] = (
        (df['close'] - df['opening_midpoint']) / df['opening_range']
    ).where(df['after_opening'])

    # Calcular factor30 (z-score del raw, por día)
    af_name = 'factor30'
    df[af_name] = df.groupby('date')['factor_raw'].transform(
        lambda x: (x - x.mean()) / x.std()
    )

    # Limpiar columnas intermedias
    df = df.drop(columns=[
        'opening_range', 'opening_midpoint', 'after_opening', 'factor_raw'
    ])

    return df, af_name

In [22]:
mnq_intraday, alpha_factor_4 = add_opening_exhaustion_factor30(mnq_intraday)

In [23]:
factores.append(alpha_factor_4)

### 1.4. Limpieza de NaN

In [24]:
mnq_model = mnq_intraday.copy()
mnq_model = mnq_model.dropna()

In [25]:
mnq_model

,date,open,high,low,close,volume,target_return_30,momentum_3,momentum_10,roc_5,...,rsi_14,stoch_k_20,bb_percent_20_15,bb_percent_30_20,price_ema30,atr_norm,reversal_momentum_factor,reversal_media_factor,reversion_vol_momentum_factor,factor30
datetime,,,,,,,,,,,,,,,,,,,,,
2019-12-23 09:30:00-05:00,2019-12-23,8733.75,8734.50,8731.25,8733.25,214,-0.000831,0.000143,0.000315,0.014315,...,49.006500,50.000000,0.739224,0.441651,-0.000051,0.000197,0.781812,-0.046414,0.877892,1.092168
2019-12-23 09:31:00-05:00,2019-12-23,8733.50,8733.50,8728.25,8729.50,1443,-0.000200,-0.000315,-0.000372,-0.020043,...,37.682151,15.625000,-0.088857,0.134612,-0.000449,0.000226,1.885030,-0.932620,9.871061,0.043107
2019-12-23 09:32:00-05:00,2019-12-23,8730.00,8730.50,8722.75,8726.00,1347,0.000029,-0.000945,-0.000601,-0.068713,...,30.579621,27.083333,-0.567670,-0.098581,-0.000795,0.000273,2.743548,-1.605686,9.872653,-0.936016
2019-12-23 09:33:00-05:00,2019-12-23,8725.75,8728.50,8723.25,8728.50,758,0.000000,-0.000544,-0.000515,-0.042944,...,39.370248,47.916667,-0.008320,0.133412,-0.000476,0.000297,2.069034,-0.936083,3.677507,-0.236642
2019-12-23 09:34:00-05:00,2019-12-23,8728.50,8729.50,8724.50,8728.25,699,0.000143,-0.000143,-0.000458,-0.068695,...,38.840581,45.833333,-0.006628,0.136863,-0.000472,0.000316,2.069099,-0.926124,3.047797,-0.306580
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:26:00-04:00,2025-06-13,21636.25,21641.75,21625.00,21629.50,3164,-0.000555,-0.000866,-0.002421,-0.199559,...,34.620111,6.617647,-0.298263,-0.049072,-0.001355,0.000797,0.526711,-1.499486,0.502966,-1.599065
2025-06-13 15:27:00-04:00,2025-06-13,21629.50,21635.00,21626.00,21629.25,1404,-0.000254,-0.000658,-0.002272,-0.137356,...,34.528096,6.250000,-0.187871,0.003740,-0.001278,0.000769,0.562791,-1.348810,0.232096,-1.602658
2025-06-13 15:28:00-04:00,2025-06-13,21629.50,21631.75,21621.25,21627.00,1947,-0.000243,-0.000439,-0.001938,-0.098160,...,33.660964,8.013937,-0.125392,0.004085,-0.001293,0.000749,0.256089,-1.347823,0.157677,-1.634997
